# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShardhaBatra/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My Rule

A content page should be reviewed if it receives high search impressions but has poor search performance or low user engagement. My rule prioritizes pages that are still visible in search results but are not attracting enough clicks, have poor average search position, or receive fewer user sessions. These pages are likely good candidates for content improvement or SEO optimization.

## Reason Codes

| Reason Code | Meaning |
|--------------|---------|
| HIGH_IMPRESSIONS | The page receives many search impressions and has the potential to attract more traffic. |
| LOW_CLICKS | The page receives fewer clicks than expected from its impressions. |
| POOR_POSITION | The page has a poor average Google search position. |
| LOW_SESSIONS | The page receives low GA4 sessions, indicating low user visits. |
| LOW_ENGAGEMENT | The page has low user engagement based on scroll events. |

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset

login(userdata.get("HF_TOKEN"))

ds = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-03/data_0.parquet",
    split="train",
    token=userdata.get("HF_TOKEN")
)

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

In [4]:
import pandas as pd

df = ds.to_pandas()

print(df.shape)

(9841378, 30)


In [5]:
baseline_df = df[
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_sessions",
        "scroll_events"
    ]
].copy()

baseline_df.head()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,NaN,NaN


In [6]:
# Signal Check 1: CTR vs Position

baseline_df["ctr"] = (
    baseline_df["gsc_clicks"] /
    baseline_df["gsc_impressions"].replace(0, pd.NA)
) * 100

signal1 = (
    baseline_df
    .groupby(pd.cut(
        baseline_df["gsc_avg_position"],
        bins=[0,10,20,50,100],
        include_lowest=True
    ))
    .agg(
        avg_ctr=("ctr","mean"),
        n=("ctr","count")
    )
)

print(signal1)

/tmp/ipykernel_4068/3905717486.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(pd.cut(


                   avg_ctr        n
gsc_avg_position                   
(-0.001, 10.0]    0.389999  2183484
(10.0, 20.0]      0.276991   519223
(20.0, 50.0]      0.163763   631491
(50.0, 100.0]     0.045227   274872


Verdict: CONFIRMED

Pages with better search positions generally have a higher CTR.
This supports using average position as one of the baseline rule signals.

In [7]:
signal2 = (
    baseline_df
    .groupby(pd.cut(
        baseline_df["gsc_impressions"],
        bins=[0,100,1000,10000,1000000],
        include_lowest=True
    ))
    .agg(
        avg_clicks=("gsc_clicks","mean"),
        n=("gsc_clicks","count")
    )
)

print(signal2)

/tmp/ipykernel_4068/95336542.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(pd.cut(


                      avg_clicks        n
gsc_impressions                          
(-0.001, 100.0]         0.018894  9207895
(100.0, 1000.0]         0.804609   601123
(1000.0, 10000.0]       4.837491    32232
(10000.0, 1000000.0]   64.593750      128


Verdict: CONFIRMED

Observation:

The bucket analysis shows that pages with higher search impressions generally receive more clicks. As impressions increase, the average number of clicks also increases. This confirms that search impressions are a useful signal for identifying high-visibility pages that may benefit from further review.

In [10]:
# Fill missing values

baseline_df["gsc_avg_position"] = baseline_df["gsc_avg_position"].fillna(0)
baseline_df["ga4_sessions"] = baseline_df["ga4_sessions"].fillna(0)
baseline_df["scroll_events"] = baseline_df["scroll_events"].fillna(0)

baseline_df.isnull().sum()

,0
report_date,0
client_hash_id,0
content_hash_id,0
gsc_impressions,0
gsc_clicks,0
gsc_avg_position,0
ga4_sessions,0
scroll_events,0
ctr,6230317


The ctr feature still contains missing values because CTR cannot be calculated when gsc_impressions is zero. Since CTR was only used for signal verification and not for the baseline scoring rule, these missing values do not affect the ranked queue.

In [11]:
# Create a rule-based score

baseline_df["score"] = 0

baseline_df.loc[baseline_df["gsc_impressions"] >= 100, "score"] += 1
baseline_df.loc[baseline_df["gsc_clicks"] <= 5, "score"] += 1
baseline_df.loc[baseline_df["gsc_avg_position"] >= 20, "score"] += 1
baseline_df.loc[baseline_df["ga4_sessions"] <= 10, "score"] += 1
baseline_df.loc[baseline_df["scroll_events"] <= 5, "score"] += 1

baseline_df[["score"]].head()

,score
0,3
1,3
2,4
3,3
4,3


In [12]:
def action(score):
    if score >= 4:
        return "Review Immediately"
    elif score >= 2:
        return "Monitor"
    else:
        return "No Action"

baseline_df["action"] = baseline_df["score"].apply(action)

baseline_df[["score", "action"]].head()

,score,action
0,3,Monitor
1,3,Monitor
2,4,Review Immediately
3,3,Monitor
4,3,Monitor


In [19]:
def reason(row):

    if row["gsc_impressions"] >= 100 and row["gsc_clicks"] <= 5:
        return "HIGH_IMPRESSIONS + LOW_CLICKS"

    elif row["gsc_avg_position"] >= 20:
        return "POOR_POSITION"

    elif row["ga4_sessions"] <= 10 and row["scroll_events"] <= 5:
        return "LOW_SESSIONS + LOW_ENGAGEMENT"

    elif row["ga4_sessions"] <= 10:
        return "LOW_SESSIONS"

    elif row["scroll_events"] <= 5:
        return "LOW_ENGAGEMENT"

    else:
        return "HIGH_IMPRESSIONS"

In [20]:
baseline_df["reason_code"] = baseline_df.apply(reason, axis=1)

In [22]:
# Rank pages by score

ranked_queue = baseline_df.sort_values(
    by="score",
    ascending=False
)

ranked_queue.head(10)

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events,ctr,score,action,reason_code
1396879,2026-03-05,client_fef1a8f436438636,content_158eb135ca99642c,162,0,23.746914,0.0,0.0,0.0,5,Review Immediately,HIGH_IMPRESSIONS + LOW_CLICKS
1396800,2026-03-05,client_fef1a8f436438636,content_eb127d86d0bc22cc,460,0,38.471739,0.0,0.0,0.0,5,Review Immediately,HIGH_IMPRESSIONS + LOW_CLICKS
1396761,2026-03-05,client_fef1a8f436438636,content_2d929abfd1e5eb64,135,0,24.444444,0.0,0.0,0.0,5,Review Immediately,HIGH_IMPRESSIONS + LOW_CLICKS
1396754,2026-03-05,client_fef1a8f436438636,content_fbf1c31cc6c27760,484,2,28.962810,0.0,0.0,0.413223,5,Review Immediately,HIGH_IMPRESSIONS + LOW_CLICKS
1396936,2026-03-05,client_fef1a8f436438636,content_93cf854771958bc6,408,0,29.620098,0.0,0.0,0.0,5,Review Immediately,HIGH_IMPRESSIONS + LOW_CLICKS
1396918,2026-03-05,client_fef1a8f436438636,content_4ab40c17e6a9a569,110,0,43.209091,0.0,0.0,0.0,5,Review Immediately,HIGH_IMPRESSIONS + LOW_CLICKS
1396717,2026-03-05,client_fef1a8f436438636,content_1ded95160e3ae724,301,2,27.338870,0.0,0.0,0.664452,5,Review Immediately,HIGH_IMPRESSIONS + LOW_CLICKS
1397092,2026-03-05,client_fef1a8f436438636,content_e4f88f3480c1b61b,214,0,25.794393,0.0,0.0,0.0,5,Review Immediately,HIGH_IMPRESSIONS + LOW_CLICKS
1397077,2026-03-05,client_fef1a8f436438636,content_6b785f1f736f94d0,173,0,36.780347,0.0,0.0,0.0,5,Review Immediately,HIGH_IMPRESSIONS + LOW_CLICKS
1397109,2026-03-05,client_fef1a8f436438636,content_07901477423d1c4b,151,0,25.152318,0.0,0.0,0.0,5,Review Immediately,HIGH_IMPRESSIONS + LOW_CLICKS


In [15]:
import os

os.makedirs("work/outputs", exist_ok=True)

ranked_queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved successfully!")

CSV saved successfully!


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [23]:
top20 = ranked_queue.head(20)

top20[
    [
        "content_hash_id",
        "score",
        "action",
        "reason_code"
    ]
]

,content_hash_id,score,action,reason_code
1396879,content_158eb135ca99642c,5,Review Immediately,HIGH_IMPRESSIONS + LOW_CLICKS
1396800,content_eb127d86d0bc22cc,5,Review Immediately,HIGH_IMPRESSIONS + LOW_CLICKS
1396761,content_2d929abfd1e5eb64,5,Review Immediately,HIGH_IMPRESSIONS + LOW_CLICKS
1396754,content_fbf1c31cc6c27760,5,Review Immediately,HIGH_IMPRESSIONS + LOW_CLICKS
1396936,content_93cf854771958bc6,5,Review Immediately,HIGH_IMPRESSIONS + LOW_CLICKS
1396918,content_4ab40c17e6a9a569,5,Review Immediately,HIGH_IMPRESSIONS + LOW_CLICKS
1396717,content_1ded95160e3ae724,5,Review Immediately,HIGH_IMPRESSIONS + LOW_CLICKS
1397092,content_e4f88f3480c1b61b,5,Review Immediately,HIGH_IMPRESSIONS + LOW_CLICKS
1397077,content_6b785f1f736f94d0,5,Review Immediately,HIGH_IMPRESSIONS + LOW_CLICKS
1397109,content_07901477423d1c4b,5,Review Immediately,HIGH_IMPRESSIONS + LOW_CLICKS


In [24]:
for i, row in top20.iterrows():
    print(f"""
Content: {row['content_hash_id']}
Action: {row['action']}
Reason: {row['reason_code']}
Confidence: High
What would make it wrong:
Missing GA4 data, temporary ranking changes, or seasonal traffic changes.
""")


Content: content_158eb135ca99642c
Action: Review Immediately
Reason: HIGH_IMPRESSIONS + LOW_CLICKS
Confidence: High
What would make it wrong:
Missing GA4 data, temporary ranking changes, or seasonal traffic changes.


Content: content_eb127d86d0bc22cc
Action: Review Immediately
Reason: HIGH_IMPRESSIONS + LOW_CLICKS
Confidence: High
What would make it wrong:
Missing GA4 data, temporary ranking changes, or seasonal traffic changes.


Content: content_2d929abfd1e5eb64
Action: Review Immediately
Reason: HIGH_IMPRESSIONS + LOW_CLICKS
Confidence: High
What would make it wrong:
Missing GA4 data, temporary ranking changes, or seasonal traffic changes.


Content: content_fbf1c31cc6c27760
Action: Review Immediately
Reason: HIGH_IMPRESSIONS + LOW_CLICKS
Confidence: High
What would make it wrong:
Missing GA4 data, temporary ranking changes, or seasonal traffic changes.


Content: content_93cf854771958bc6
Action: Review Immediately
Reason: HIGH_IMPRESSIONS + LOW_CLICKS
Confidence: High
What would 

Top-20 Review

The highest ranked pages mostly receive the action Review Immediately because they satisfy multiple baseline conditions such as high impressions, low clicks, poor search position, or low engagement. These pages should be reviewed first, but manual inspection is still necessary because missing GA4 data, temporary ranking fluctuations, or seasonal traffic changes may affect the score.

| Page | Action             | Reason Code   | Confidence | What would make it wrong                       |
| ---- | ------------------ | ------------- | ---------- | ---------------------------------------------- |
| 1    | Review Immediately | LOW_SESSIONS  | High       | Missing GA4 data could underestimate sessions. |
| 2    | Review Immediately | POOR_POSITION | High       | Temporary ranking fluctuations.                |
| 3    | Review Immediately | LOW_CLICKS    | Medium     | Seasonal changes in search demand.             |


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Some pages may receive a high score because GA4 sessions or scroll events are unavailable rather than because the content is truly declining. These pages should be reviewed manually before taking action.

## Leakage Check

I confirmed that my baseline rule does not use any future-window information, label-derived columns, or product decision flags. The rule only uses historical search and engagement signals available at the decision time.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.